In [ ]:
!pip -q install amplpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 27.5 MB/s eta 0:00:00


In [ ]:
# 1. Instalación e inicialización de AMPL con tu licencia Community Edition
from amplpy import ampl_notebook

# Inicializa AMPL en Colab con tu UUID (Community Edition) Inicializar AMPL
ampl = ampl_notebook(
    modules=["coin", "highs", "cbc", "gurobi", "cplex"],  # solvers disponibles
    license_uuid="936b618d-a013-406f-9809-49679f557c26"
)

Licensed to AMPL Academic Community Edition License for <m.godoyseplveda@uandresbello.edu>.


In [ ]:
%%writefile 12_7_4.dat
param a1 := 2;      # coeficientes lineales
param a2 := 3;

param b1 := 1;      # coeficientes cuadráticos (f = a·x − b·x²)
param b2 := 1;

param R  := 2;      # x1 + x2 ≤ 2


Overwriting 12_7_4.dat


In [ ]:
%%writefile 12_7_4_nl.mod
param a1; param a2; param b1; param b2; param R;

var x1 >= 0;
var x2 >= 0;

maximize Z:
      a1*x1 + a2*x2 - b1*x1^2 - b2*x2^2;

s.t. Cap:  x1 + x2 <= R;

Overwriting 12_7_4_nl.mod


In [ ]:
ampl.read('12_7_4_nl.mod')
ampl.readData('12_7_4.dat')

ampl.option['solver'] = 'ipopt'
ampl.option['solver_msg'] = 0   # silencio

ampl.solve()
print("QP (IPOPT):")
ampl.display('x1','x2','Z')

Ipopt 3.12.13: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt
******************************************************************************

This is Ipopt version 3.12.13, running with linear solver mumps.
NOTE: Other linear solvers might be more efficient (see Ipopt documentation).

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:        2
Number of nonzeros in Lagrangian Hessian.............:        2

Total number of variables............................:        2
                     variables with only lower bounds:        2
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Tot

In [ ]:
%%writefile 12_7_4_wolfe.mod
param a1;
param a2;
param b1;
param b2;
param R;

# Variables originales
var x1 >= 0;
var x2 >= 0;

# Gradientes linealizados
var w1 >= 0;
var w2 >= 0;

# Variable valor objetivo
var theta;

maximize Obj: theta;

# Restriccion de soporte (KKT linealizado)
s.t. KKT:
    theta <= a1*x1 + a2*x2 - w1*x1 - w2*x2;

# Complementariedad
s.t. Link1: w1 = x1;
s.t. Link2: w2 = x2;

# Restriccion original (capacidad)
s.t. Cap:   w1 + w2 <= R;      # equivalente a x1 + x2 <= R


Overwriting 12_7_4_wolfe.mod


In [ ]:
ampl.reset()
ampl.read('12_7_4_wolfe.mod')
ampl.readData('12_7_4.dat')

ampl.option['solver'] = 'couenne'
ampl.option['solver_msg'] = 0
ampl.solve()
print("Modelo lineal (Wolfe):")
ampl.display('x1','x2','theta')

Couenne 0.5.8 -- an Open-Source solver for Mixed Integer Nonlinear Optimization
Mailing list: couenne@list.coin-or.org
Instructions: http://www.coin-or.org/Couenne
couenne: 
ANALYSIS TEST: Couenne: new cutoff value -2.0000000000e+00 (0.007969 seconds)
NLP0012I 
              Num      Status      Obj             It       time                 Location
NLP0014I             1         OPT -3.125       10 0.013006
Couenne: new cutoff value -3.1250000200e+00 (0.022067 seconds)
Loaded instance "/tmp/at1914.nl"
Constraints:            4
Variables:              5 (0 integer)
Auxiliaries:            6 (0 integer)

Coin0506I Presolve 12 (-2) rows, 5 (-6) columns and 27 (-4) elements
Clp0006I 0  Obj -3.1245875 Primal inf 1.0415282 (1) Dual inf 2.999999 (1)
Clp0006I 7  Obj -3.25
Clp0000I Optimal - objective value -3.25
Clp0032I Optimal objective -3.25 - 7 iterations time 0.002, Presolve 0.00
Clp0000I Optimal - objective value -3.25
Cbc0012I Integer solution of -3.125 found by Couenne Rounding NLP af